# Mushroom — Beslisboom depth 5

Eén modeltype per notebook. Oorspronkelijke Mushroom-modellen: Andrew; de gedeelde
vergelijking en deze reproduceerbare versie: Codex op verzoek van Jorre.
Jorre rondt de definitieve taakverdeling en GenAI-policy later af (stap 5).

De huidige referentie is de correct uitgevoerde vergelijking op dezelfde 4.100
trainingsrijen, 900 ontwikkeltestrecords en vijf folds. Het model wordt hieronder
werkelijk opnieuw gefit met de vastgelegde configuratie. Alle preprocessing wordt
binnen de fold geleerd. De ontwikkeltest is eerder bekeken en is geen nieuwe
onafhankelijke eindtest. Dit notebook wijzigt de website of bronstudie niet.

We laden de vastgelegde parameters en leggen uit waarom dit model is opgenomen.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
summary = json.loads((ROOT/'MushroomFinal/comparison_correct/comparison_summary.json').read_text(encoding='utf-8'))
key = 'tree_baseline'
display(pd.Series(summary['classifier_parameters'][key], name='Vastgelegde parameters'))


ccp_alpha                    0.0
class_weight                None
criterion                   gini
max_depth                      5
max_features                None
max_leaf_nodes              None
min_impurity_decrease        0.0
min_samples_leaf               1
min_samples_split              2
min_weight_fraction_leaf     0.0
monotonic_cst               None
random_state                  42
splitter                    best
Name: Vastgelegde parameters, dtype: object

Een kleine beslisboom vormt een snelle, interpreteerbare referentie; maximale diepte 5 begrenst complexiteit.

We fitten vijf trainingsfolds en daarna één model op alle trainingsrijen. De code controleert alle 900 labels en kansen tegen de opgeslagen referentie.

In [2]:
from Final.tools.model_workflow import reproduce
result = reproduce('tree_baseline')
display(pd.DataFrame(result['folds']))
display(pd.Series({k:v for k,v in result.items() if k != 'folds'}))
assert result['labels_match_original'] and result['max_probability_difference'] < 1e-12


tree_baseline fold 1 /5


tree_baseline fold 2 /5


tree_baseline fold 3 /5


tree_baseline fold 4 /5


tree_baseline fold 5 /5


,fold,accuracy,balanced_accuracy,macro_f1,precision_p,recall_p,f1_p,roc_auc,average_precision,tn,fp,fn,tp
0,1,0.662195,0.576551,0.553204,0.663462,0.221865,0.332530,0.606792,0.489129,474,35,242,69
1,2,0.684146,0.596109,0.573971,0.782609,0.231511,0.357320,0.633769,0.516667,489,20,239,72
2,3,0.695122,0.607527,0.589461,0.819149,0.248387,0.381188,0.631996,0.524172,493,17,233,77
3,4,0.704878,0.624225,0.615116,0.798246,0.293548,0.429245,0.638466,0.549961,487,23,219,91
4,5,0.680488,0.593232,0.572590,0.744898,0.235484,0.357843,0.627660,0.526799,485,25,237,73


model                         tree_baseline
cv_macro_f1_mean                   0.580869
cv_macro_f1_std                    0.023066
cv_recall_p_mean                   0.246159
accuracy                                0.7
balanced_accuracy                  0.609824
macro_f1                           0.588816
precision_p                         0.89011
recall_p                           0.237537
f1_p                                  0.375
roc_auc                            0.652616
average_precision                  0.554699
tn                                      549
fp                                       10
fn                                      260
tp                                       81
fit_and_cv_seconds                 0.154146
training_rows                          4100
test_rows                               900
labels_match_original                  True
max_probability_difference              0.0
sklearn                               1.4.2
dtype: object

De uiteindelijke keuze staat in `09_vergelijking.ipynb`. Een hoge score voor de eetbare klasse is geen bevestiging van eetbaarheid; de website gebruikt een afzonderlijke voorzichtige beslisregel.